# Silver Layer Transformation

## Objective
This notebook reads the Bronze Delta table, performs data quality validation,
standardizes data types and text fields, validates investment amounts,
and writes the cleaned dataset into the Silver layer. 

In [0]:
account_key=dbutils.secrets.get(scope = "My_scope", key = "DB-secret")
# plx = dbutils.secrets.get(scope = "my_scope", key = "sa")
# print(plx)
# dbutils.secrets.listScopes()

In [0]:
storage_account = "storagestartup"
container = "startup-data"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    account_key
)

In [0]:
bronze_df = spark.read \
    .format("delta") \
    .load("abfss://startup-data@storagestartup.dfs.core.windows.net/bronze/delta/startup_funding")



#Data Inspection

In [0]:
print("Total Rows:", bronze_df.count())

Total Rows: 1100


In [0]:
print("Total Columns:", len(bronze_df.columns))

Total Columns: 8


In [0]:
bronze_df.printSchema()

root
 |-- Startup: string (nullable = true)
 |-- Industry: string (nullable = true)
 |-- SubVertical: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Investors: string (nullable = true)
 |-- InvestmentType: string (nullable = true)
 |-- InvestmentAmount_USD: integer (nullable = true)
 |-- Date: string (nullable = true)



In [0]:
display(bronze_df.limit(10))

Startup,Industry,SubVertical,City,Investors,InvestmentType,InvestmentAmount_USD,Date
Housejoy,EdTech,K12,Mumbai,Lightspeed India,Seed,199000,19-04-2023
Groww,Media,Streaming,Bengaluru,IFC,Seed,1668000,28-01-2025
Groww,Mobility,Ride Sharing,Hyderabad,"Nexus Venture Partners, Peak XV",Series B,38052000,14-03-2021
FarmBox,Consumer Electronics,Wearables,Gurugram,"Kalaari Capital, Y Combinator",Seed,455000,11/9/2023
Udaan,RealEstate,Rental Tech,Mumbai,Bessemer Venture Partners,Seed,89000,31-01-2024
AeroStack,EdTech,Coding Bootcamp,Delhi,"Matrix Partners India, Peak XV",Pre-Series A,143000,24-07-2021
Ola,Retail,Fashion,Mumbai,Tiger Global Management,Seed,62000,12/1/2023
FinSpace,Consumer Electronics,Wearables,Gurugram,Blume Ventures,Growth,491721000,22-01-2025
Rapido,FoodTech,Food Delivery,Kolkata,"A91 Partners, Kedaara Capital, Tiger Global Management",Seed,459000,27-11-2023
HyperLoop,EdTech,Test Prep,Ahmedabad,"Mirae Asset, SoftBank Vision Fund, Tiger Global",Series B,35610000,4/5/2020


## Data Quality Validation
### Null Value Check

In [0]:
from pyspark.sql.functions import col, sum, when

bronze_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in bronze_df.columns
]).show()

+-------+--------+-----------+----+---------+--------------+--------------------+----+
|Startup|Industry|SubVertical|City|Investors|InvestmentType|InvestmentAmount_USD|Date|
+-------+--------+-----------+----+---------+--------------+--------------------+----+
|      0|       0|          0|   0|        0|             0|                   0|   0|
+-------+--------+-----------+----+---------+--------------+--------------------+----+



### Duplicate Record Validation

In [0]:
total_records = bronze_df.count()
unique_records = bronze_df.dropDuplicates().count()

print(f"Total Records: {total_records}")
print(f"Unique Records: {unique_records}")
print(f"Duplicate Records: {total_records - unique_records}")

Total Records: 1100
Unique Records: 1100
Duplicate Records: 0


Result:
No duplicate records were detected in the Bronze dataset.

### Data Standardization
### Data Type Conversion

In [0]:
from pyspark.sql.functions import col, to_date

silver_df = bronze_df \
    .withColumn("InvestmentAmount_USD", col("InvestmentAmount_USD").cast("double")) 

In [0]:
silver_df.printSchema()

root
 |-- Startup: string (nullable = true)
 |-- Industry: string (nullable = true)
 |-- SubVertical: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Investors: string (nullable = true)
 |-- InvestmentType: string (nullable = true)
 |-- InvestmentAmount_USD: double (nullable = true)
 |-- Date: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, regexp_replace, to_date

silver_df = bronze_df \
    .withColumn(
        "Date",
        regexp_replace(col("Date"), "/", "-")
    ) \
    .withColumn(
        "Date",
        to_date(col("Date"), "d-M-yyyy")
    ) \
    .withColumn(
        "InvestmentAmount_USD",
        col("InvestmentAmount_USD").cast("double")
    )

In [0]:
silver_df.printSchema()

root
 |-- Startup: string (nullable = true)
 |-- Industry: string (nullable = true)
 |-- SubVertical: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Investors: string (nullable = true)
 |-- InvestmentType: string (nullable = true)
 |-- InvestmentAmount_USD: double (nullable = true)
 |-- Date: date (nullable = true)



Standardize Text Columns

In [0]:
from pyspark.sql.functions import trim

silver_df = silver_df \
    .withColumn("Startup", trim(col("Startup"))) \
    .withColumn("Industry", trim(col("Industry"))) \
    .withColumn("SubVertical", trim(col("SubVertical"))) \
    .withColumn("City", trim(col("City"))) \
    .withColumn("Investors", trim(col("Investors"))) \
    .withColumn("InvestmentType", trim(col("InvestmentType")))

Check Investment Amount Quality (INR/USD Mix)

In [0]:
silver_df.select("InvestmentAmount_USD").show(30, False)

+--------------------+
|InvestmentAmount_USD|
+--------------------+
|199000.0            |
|1668000.0           |
|3.8052E7            |
|455000.0            |
|89000.0             |
|143000.0            |
|62000.0             |
|4.91721E8           |
|459000.0            |
|3.561E7             |
|54000.0             |
|933000.0            |
|680000.0            |
|374000.0            |
|1.51993E8           |
|1643000.0           |
|2.01485E8           |
|4142000.0           |
|54000.0             |
|1.1211E7            |
|5243000.0           |
|84000.0             |
|12000.0             |
|5.3939E7            |
|269000.0            |
|905000.0            |
|104000.0            |
|333000.0            |
|29000.0             |
|111000.0            |
+--------------------+
only showing top 30 rows


In [0]:
from pyspark.sql.functions import min, max

silver_df.select(
    min("InvestmentAmount_USD").alias("Min"),
    max("InvestmentAmount_USD").alias("Max")
).show()

+------+---------+
|   Min|      Max|
+------+---------+
|5000.0|5.74676E8|
+------+---------+



In [0]:
silver_df.select(
    "InvestmentAmount_USD"
).distinct().orderBy("InvestmentAmount_USD").show(50, False)

+--------------------+
|InvestmentAmount_USD|
+--------------------+
|5000.0              |
|6000.0              |
|7000.0              |
|8000.0              |
|9000.0              |
|10000.0             |
|12000.0             |
|13000.0             |
|14000.0             |
|15000.0             |
|16000.0             |
|17000.0             |
|18000.0             |
|19000.0             |
|20000.0             |
|21000.0             |
|23000.0             |
|24000.0             |
|25000.0             |
|26000.0             |
|27000.0             |
|29000.0             |
|32000.0             |
|33000.0             |
|34000.0             |
|36000.0             |
|37000.0             |
|38000.0             |
|41000.0             |
|42000.0             |
|43000.0             |
|46000.0             |
|47000.0             |
|50000.0             |
|51000.0             |
|52000.0             |
|53000.0             |
|54000.0             |
|55000.0             |
|56000.0             |
|57000.0   

In [0]:
silver_df.filter(
    col("InvestmentAmount_USD").cast("string").rlike("₹|USD|INR|\\$|,")
).show(truncate=False)

+-------+--------+-----------+----+---------+--------------+--------------------+----+
|Startup|Industry|SubVertical|City|Investors|InvestmentType|InvestmentAmount_USD|Date|
+-------+--------+-----------+----+---------+--------------+--------------------+----+
+-------+--------+-----------+----+---------+--------------+--------------------+----+



Result:
The InvestmentAmount_USD column contains only numeric values.
No currency symbols or malformed values were found.

#City normalization

In [0]:
silver_df.select("City").distinct().orderBy("City").show(100, False)

+---------+
|City     |
+---------+
|Ahmedabad|
|Bengaluru|
|Chennai  |
|Delhi    |
|Gurugram |
|Hyderabad|
|Kolkata  |
|Mumbai   |
|Noida    |
|Pune     |
+---------+



City normalization was validated. No inconsistent city names were found; therefore, no transformations were required.

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://startup-data@storagestartup.dfs.core.windows.net/silver/delta/startup_funding")

In [0]:
silver_check = spark.read.format("delta").load(
    "abfss://startup-data@storagestartup.dfs.core.windows.net/silver/delta/startup_funding"
)

display(silver_check.limit(5))
print("Silver Rows:", silver_check.count())
print("Silver Columns:", len(silver_check.columns))

Startup,Industry,SubVertical,City,Investors,InvestmentType,InvestmentAmount_USD,Date
Housejoy,EdTech,K12,Mumbai,Lightspeed India,Seed,199000.0,2023-04-19
Groww,Media,Streaming,Bengaluru,IFC,Seed,1668000.0,2025-01-28
Groww,Mobility,Ride Sharing,Hyderabad,"Nexus Venture Partners, Peak XV",Series B,3.8052E7,2021-03-14
FarmBox,Consumer Electronics,Wearables,Gurugram,"Kalaari Capital, Y Combinator",Seed,455000.0,2023-09-11
Udaan,RealEstate,Rental Tech,Mumbai,Bessemer Venture Partners,Seed,89000.0,2024-01-31


Silver Rows: 1100
Silver Columns: 8


# SUMMARY
->The Silver_Transformation notebook transforms the raw Bronze data into a clean, standardized, and analytics-ready dataset.

 -> Data quality operations were performed, including handling missing values, removing duplicate records, standardizing city names and funding stages, converting date fields to the appropriate format, and normalizing funding amounts. 
 
 ->Additional derived columns were created where required to support downstream analysis.

 -> The cleansed data was then stored as a Delta table, providing a reliable and consistent foundation for the Gold layer, where business-focused analytics and aggregations are generated.